# Input-output fundamentals and carbon tiers

Leontief accounting followed by direct and indirect carbon-intensity decomposition.

Run the notebook from top to bottom. Local section numbers are retained within each part.

1. [08p: EEIO Input-Output Fundamentals](#part-08p)
2. [08q: EEIO Carbon-Intensity Tier Decomposition](#part-08q)


In [1]:
import numpy as np
import pandas as pd

<a id="part-08p"></a>

## Part 1 — 08p: EEIO Input-Output Fundamentals

This notebook opens the Environmentally-Extended Input-Output (EEIO)
carbon-accounting material with six self-contained worked examples on
small, fully hardcoded synthetic economies (3-6 sectors, sometimes with
a regional dimension) — no external data files are loaded anywhere in
this notebook.

**On this material's scope, disclosed once here and referenced from
every subsequent notebook in this sequence:** a meaningful portion of
the EEIO carbon-accounting exercises this sequence could otherwise
build depend on data that simply is not available in this environment
— a full multi-region input-output table, the country/sector/emissions
reference files behind an extended carbon-exposure family of exercises,
and the country-label files behind an OECD embodied-carbon database.
Those pieces are not built here. A further set of exercises depends on
the same missing multi-region matrices but is covered in a later
notebook (`08t`), which displays precomputed result tables directly
rather than re-deriving them, with that choice clearly disclosed there.


In [2]:
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### Question 1 — Technical-coefficient matrix and the Leontief inverse

A 4-sector toy economy: intermediate-consumption matrix $Z$, final
demand $y$, and total output $x$ (with $x = Z\mathbf{1} + y$ verified).
The technical-coefficient matrix $A = Z \, \text{diag}(x)^{-1}$ gives the
input required from each sector per unit of output of the *column*
sector; the Leontief inverse $L=(I-A)^{-1}$ recovers $x$ from $y$
exactly, and propagates a final-demand shock $\Delta y$ into the output
response $\Delta x = L\,\Delta y$ across every sector, not just the one
directly demanded.


In [3]:
Z1 = np.array(
    [
        [500, 800, 1600, 1250],
        [500, 400, 1600, 625],
        [250, 800, 2400, 1250],
        [100, 200, 800, 4375],
    ],
    dtype=float,
)
y1 = np.array([850, 875, 3300, 7025], dtype=float)
x1 = np.array([5000, 4000, 8000, 12500], dtype=float)
sectors4 = ["Energy", "Materials", "Industrials", "Services"]

x1_check = Z1.sum(axis=1) + y1
print("x vs. Z*1 + y:")
display(pd.DataFrame({"x": x1, "x_check": x1_check}, index=sectors4))

A1 = Z1 / x1[None, :]
print("\nA (technical-coefficient matrix):")
display(pd.DataFrame(A1, index=sectors4, columns=sectors4))

I4 = np.eye(4)
L1 = np.linalg.inv(I4 - A1)
x1_from_L = L1 @ y1
print("\nx recovered via L*y:")
display(pd.DataFrame({"x": x1, "L@y": x1_from_L}, index=sectors4))

print("\nLeontief inverse L:")
display(pd.DataFrame(L1, index=sectors4, columns=sectors4))

delta_y1 = np.array([0, 0, 0, 10.0])
delta_x1 = L1 @ delta_y1
print("\nOutput response to a +10 final-demand shock in Services:")
display(pd.DataFrame({"Delta_x": delta_x1}, index=sectors4))

x vs. Z*1 + y:


,x,x_check
Energy,"5,000.0000","5,000.0000"
Materials,"4,000.0000","4,000.0000"
Industrials,"8,000.0000","8,000.0000"
Services,"12,500.0000","12,500.0000"



A (technical-coefficient matrix):


,Energy,Materials,Industrials,Services
Energy,0.1000,0.2000,0.2000,0.1000
Materials,0.1000,0.1000,0.2000,0.0500
Industrials,0.0500,0.2000,0.3000,0.1000
Services,0.0200,0.0500,0.1000,0.3500



x recovered via L*y:


,x,L@y
Energy,"5,000.0000","5,000.0000"
Materials,"4,000.0000","4,000.0000"
Industrials,"8,000.0000","8,000.0000"
Services,"12,500.0000","12,500.0000"



Leontief inverse L:


,Energy,Materials,Industrials,Services
Energy,1.1881,0.3894,0.4919,0.2884
Materials,0.1678,1.2552,0.4336,0.1891
Industrials,0.1430,0.4110,1.6303,0.3044
Services,0.0715,0.1718,0.2993,1.6087



Output response to a +10 final-demand shock in Services:


,Delta_x
Energy,2.8842
Materials,1.8907
Industrials,3.0444
Services,16.0872


### Question 2 — The dual price system

Same 4-sector economy, now with a value-added matrix $V$ split across
two primary-factor rows (labor, capital say) and a price-weighting
vector $\psi$. The value-added-coefficient matrix $B = V \, \text{diag}(x)^{-1}$
and the *dual* Leontief inverse $\tilde{L} = (I-A')^{-1}$ give the price
vector $p = \tilde{L}\,\upsilon$ where $\upsilon = B'\psi$ — verified
against the accounting identity $p = A'p + B'\psi$ and the aggregate
value identity $y'p = x'\upsilon$. A cost shock in one primary input
($\Delta\upsilon$, energy here) then propagates into sector-level and
aggregate (PPI/CPI-style) price inflation.


In [4]:
V2 = np.array(
    [
        [3000, 800, 1000, 3000],
        [650, 1000, 600, 2000],
    ],
    dtype=float,
)
psi2 = np.array([1.0, 1.0])

V_check = x1 - Z1.sum(axis=0)
print("Value added (row sum of V) vs. x - Z'*1:")
display(pd.DataFrame({"sum(V)": V2.sum(axis=0), "x - Z'*1": V_check}, index=sectors4))

B2 = V2 / x1[None, :]
upsilon2 = B2.T @ psi2
dual_L2 = np.linalg.inv(I4 - A1.T)
p2 = dual_L2 @ upsilon2
print("\nPrice vector p and the accounting identity A'p + B'psi:")
display(pd.DataFrame({"p": p2, "A'p + B'psi": A1.T @ p2 + B2.T @ psi2}, index=sectors4))

print(
    f"\ny'p = {y1 @ p2:,.2f}   x'upsilon = {x1 @ upsilon2:,.2f}   "
    f"(difference: {y1 @ p2 - x1 @ upsilon2:.2e})"
)

upsilon2b = V2.sum(axis=0) / x1
p2b = dual_L2 @ upsilon2b
print(
    "\nAlternative upsilon (total value added per unit output) and its price vector "
    "(both normalizations give p = 1, the base-year price level, by construction):"
)
display(pd.DataFrame({"upsilon": upsilon2b, "p": p2b}, index=sectors4))

delta_upsilon2 = np.array([0.10, 0, 0, 0])
delta_p2 = dual_L2 @ delta_upsilon2
print("\nPrice response (%) to a +10% cost shock in Energy:")
display(pd.DataFrame({"Delta_p (%)": 100 * delta_p2}, index=sectors4))

alpha_ppi = x1 / x1.sum()
ppi2 = alpha_ppi @ delta_p2
alpha_cpi = y1 / y1.sum()
cpi2 = alpha_cpi @ delta_p2
print(
    f"\nOutput-weighted (PPI-style) inflation: {100 * ppi2:.2f}%   "
    f"Final-demand-weighted (CPI-style) inflation: {100 * cpi2:.2f}%"
)

Value added (row sum of V) vs. x - Z'*1:


,sum(V),x - Z'*1
Energy,"3,650.0000","3,650.0000"
Materials,"1,800.0000","1,800.0000"
Industrials,"1,600.0000","1,600.0000"
Services,"5,000.0000","5,000.0000"



Price vector p and the accounting identity A'p + B'psi:


,p,A'p + B'psi
Energy,1.0000,1.0000
Materials,1.0000,1.0000
Industrials,1.0000,1.0000
Services,1.0000,1.0000



y'p = 12,050.00   x'upsilon = 12,050.00   (difference: 3.64e-12)

Alternative upsilon (total value added per unit output) and its price vector (both normalizations give p = 1, the base-year price level, by construction):


,upsilon,p
Energy,0.7300,1.0000
Materials,0.4500,1.0000
Industrials,0.2000,1.0000
Services,0.4000,1.0000



Price response (%) to a +10% cost shock in Energy:


,Delta_p (%)
Energy,11.8811
Materials,3.8936
Industrials,4.9191
Services,2.8842



Output-weighted (PPI-style) inflation: 5.10%   Final-demand-weighted (CPI-style) inflation: 4.15%


### Question 3 — Geometric-series convergence of the Leontief inverse

The same technical-coefficient matrix $A$, illustrating why
$L=(I-A)^{-1} = \sum_{k=0}^{\infty} A^k$: each round $y_k = A^k y$
represents the $k$-th tier of indirect output requirements (round-1
suppliers' own suppliers, and so on), shrinking geometrically since $A$'s
spectral radius is below 1. The cumulative sum $\sum_{j=0}^{k} y_j$
converges to $Ly$ as $k$ grows — reproduced at a representative set of
snapshot rounds (0-5, 10, 20, 25).


In [5]:
nIters = 30
y_k = np.zeros((4, nIters + 1))
for k in range(nIters + 1):
    y_k[:, k] = np.linalg.matrix_power(A1, k) @ y1
yc_k = np.cumsum(y_k, axis=1)

snapshot_k = [0, 1, 2, 3, 4, 5, 10, 20, 25]
snapshot_df = pd.DataFrame(
    yc_k[:, snapshot_k], index=sectors4, columns=[f"k={k}" for k in snapshot_k]
)
print("Cumulative output requirement sum_{j=0}^{k} A^j y, at snapshot rounds:")
display(snapshot_df)

L1_check = np.linalg.inv(I4 - A1)
print(
    f"\nAt k=30 the cumulative sum is within "
    f"{np.max(np.abs(yc_k[:, -1] - L1_check @ y1)):.2e} of the exact L@y "
    "(geometric convergence, not yet exact at a finite number of rounds)."
)

Cumulative output requirement sum_{j=0}^{k} A^j y, at snapshot rounds:


,k=0,k=1,k=2,k=3,k=4,k=5,k=10,k=20,k=25
Energy,850.0000,"2,472.5000","3,538.4500","4,169.2262","4,531.2841","4,736.5032","4,985.4165","4,999.9556","4,999.9976"
Materials,875.0000,"2,058.7500","2,863.8500","3,350.1181","3,632.1920","3,792.9069","3,988.5208","3,999.9651","3,999.9981"
Industrials,"3,300.0000","5,210.0000","6,385.8250","7,080.8862","7,480.9114","7,708.0301","7,983.8322","7,999.9508","7,999.9973"
Services,"7,025.0000","9,874.5000","11,154.4625","11,781.6059","12,107.5411","12,282.9658","12,488.1815","12,499.9641","12,499.9980"



At k=30 the cumulative sum is within 1.50e-04 of the exact L@y (geometric convergence, not yet exact at a finite number of rounds).


### Question 4 — A small carbon-footprint worked example

A 3-sector toy economy with a 2-row emissions matrix $CE$ (say CO$_2$
and CH$_4$), computed via `eeio_compute_impact1`: direct emissions
intensity $D_x = CE/x'$, total (production-based) intensity
$D_y = D_x L$, and the resulting total footprint $\varpi_y = D_y y$ —
cross-checked against $\varpi_x = D_y(I-A)x$ (an equivalent,
production-side view of the same total). $C_y$ breaks the footprint down
by both emitting sector and final-demand sector simultaneously,
summing back to $\varpi_y$. Two variants differing only in the
$(y, x, CE)$ inputs are run in a single loop.


In [6]:
def eeio_compute_impact1(Z, y, x, CE):
    n = len(y)
    D_x = CE / x[None, :]
    A = Z / x[None, :]
    L = np.linalg.inv(np.eye(n) - A)
    x_from_L = L @ y
    D_y = D_x @ L
    varpi_y = D_y @ y
    C_y = D_y * y[None, :]
    varpi_x = D_y @ (np.eye(n) - A) @ x_from_L
    return dict(D_x=D_x, D_y=D_y, varpi_x=varpi_x, varpi_y=varpi_y, C_y=C_y, A=A, L=L, x=x_from_L)


Z3 = np.array([[100, 300, 100], [250, 150, 200], [25, 200, 75]], dtype=float)
sectors3 = ["Sector 1", "Sector 2", "Sector 3"]
species2 = ["CO2", "CH4"]

IOT5_CASES = {
    "iot5a": dict(
        y=np.array([500, 1400, 200.0]),
        x=np.array([1000, 2000, 500.0]),
        CE=np.array([[50, 20, 5.0], [3, 1, 0.0]]),
    ),
    "iot5b": dict(
        y=np.array([0, 1400, 200.0]),
        x=np.array([500, 2000, 500.0]),
        CE=np.array([[25, 20, 5.0], [1.5, 1, 0.0]]),
    ),
}

for name, case in IOT5_CASES.items():
    r = eeio_compute_impact1(Z3, case["y"], case["x"], case["CE"])
    print(f"--- {name} ---")
    display(pd.DataFrame({"x (input)": case["x"], "x (recovered via L)": r["x"]}, index=sectors3))
    print("A:")
    display(pd.DataFrame(r["A"], index=sectors3, columns=sectors3))
    print("Direct intensity D_x and total intensity D_y:")
    display(
        pd.DataFrame(
            {"D_x_" + s: r["D_x"][i] for i, s in enumerate(species2)}, index=sectors3
        ).join(
            pd.DataFrame({"D_y_" + s: r["D_y"][i] for i, s in enumerate(species2)}, index=sectors3)
        )
    )
    print(
        f"Total footprint varpi_y = {dict(zip(species2, r['varpi_y']))}   "
        f"varpi_x (production-side check) = {dict(zip(species2, r['varpi_x']))}"
    )
    print("C_y (footprint by emitting sector x final-demand sector), row sums = varpi_y:")
    display(
        pd.DataFrame(r["C_y"], index=[f"emits ({s})" for s in species2], columns=sectors3).assign(
            row_sum=r["C_y"].sum(axis=1)
        )
    )
    print()

--- iot5a ---


,x (input),x (recovered via L)
Sector 1,"1,000.0000","1,000.0000"
Sector 2,"2,000.0000","2,000.0000"
Sector 3,500.0000,500.0000


A:


,Sector 1,Sector 2,Sector 3
Sector 1,0.1000,0.1500,0.2000
Sector 2,0.2500,0.0750,0.4000
Sector 3,0.0250,0.1000,0.1500


Direct intensity D_x and total intensity D_y:


...

,Sector 1,Sector 2,Sector 3
Sector 1,0.2000,0.1500,0.2000
Sector 2,0.5000,0.0750,0.4000
Sector 3,0.0500,0.1000,0.1500


Direct intensity D_x and total intensity D_y:


,D_x_CO2,D_x_CH4,D_y_CO2,D_y_CH4
Sector 1,0.0500,0.0030,0.0836,0.0048
Sector 2,0.0100,0.0005,0.0293,0.0015
Sector 3,0.0100,0.0000,0.0452,0.0018


Total footprint varpi_y = {'CO2': np.float64(49.99999999999999), 'CH4': np.float64(2.5)}   varpi_x (production-side check) = {'CO2': np.float64(49.99999999999999), 'CH4': np.float64(2.5)}
C_y (footprint by emitting sector x final-demand sector), row sums = varpi_y:


,Sector 1,Sector 2,Sector 3,row_sum
emits (CO2),0.0000,40.9589,9.0411,50.0000
emits (CH4),0.0000,2.1301,0.3699,2.5000


### Question 5 — A multi-region toy accounting table

A 6-sector economy split across 3 implicit "regions" (final demand $y$
has 3 columns, one per region), with a single emissions vector $CE$.
Using the *dual* (transposed) Leontief inverse to allocate total
emissions intensity to final demand, `CE_y[i,r]` gives sector $i$'s
emissions embedded in region $r$'s consumption — summed across all 6
sectors to get each region's total consumption-based carbon
footprint.


In [7]:
Z6 = np.array(
    [
        [100, 300, 10, 10, 20, 0],
        [250, 150, 20, 0, 10, 0],
        [10, 10, 110, 310, 0, 0],
        [20, 20, 80, 25, 15, 20],
        [10, 5, 8, 3, 40, 7],
        [5, 2, 8, 8, 12, 35],
    ],
    dtype=float,
)
y6 = np.array(
    [
        [500, 200, 25],
        [800, 100, 17],
        [20, 200, 15],
        [0, 200, 5],
        [5, 25, 50],
        [3, 50, 50],
    ],
    dtype=float,
)
x6 = np.array([1165, 1347, 675, 385, 153, 173], dtype=float)
CE6 = 1000 * np.array([50, 20, 10, 10, 5, 5], dtype=float)
sectors6 = [f"S{i + 1}" for i in range(6)]
regions3 = ["Region 1", "Region 2", "Region 3"]

A6 = Z6 / x6[None, :]
CI6 = CE6 / x6
L6_prime = np.linalg.inv(np.eye(6) - A6).T
CI6_y = L6_prime @ CI6
CE6_y = CI6_y[:, None] * y6

print("Total emissions embedded in each region's final demand, by sector:")
display(pd.DataFrame(CE6_y, index=sectors6, columns=regions3).assign(row_sum=CE6_y.sum(axis=1)))

region_totals = CE6_y.sum(axis=0)
print("\nEach region's total consumption-based carbon footprint:")
display(pd.DataFrame({"total_CE": region_totals}, index=regions3))

Total emissions embedded in each region's final demand, by sector:


,Region 1,Region 2,Region 3,row_sum
S1,"28,397.4037","11,358.9615","1,419.8702","41,176.2354"
S2,"26,002.6494","3,250.3312",552.5563,"29,805.5368"
S3,590.1826,"5,901.8264",442.6370,"6,934.6460"
S4,0.0000,"11,282.5647",282.0641,"11,564.6288"
S5,348.7820,"1,743.9102","3,487.8203","5,580.5125"
S6,143.8381,"2,397.3012","2,397.3012","4,938.4405"



Each region's total consumption-based carbon footprint:


,total_CE
Region 1,"55,482.8558"
Region 2,"35,934.8951"
Region 3,"8,582.2491"


### Sanity checks


In [8]:
checks = []
checks.append(
    (
        "Q1: x = Z*1 + y exactly, and the Leontief inverse recovers the same x from y",
        np.allclose(x1_check, x1) and np.allclose(x1_from_L, x1),
    )
)
checks.append(
    (
        "Q2: the dual-price accounting identity p = A'p + B'psi holds",
        np.allclose(A1.T @ p2 + B2.T @ psi2, p2),
    )
)
checks.append(
    ("Q2: the aggregate value identity y'p = x'upsilon holds", abs(y1 @ p2 - x1 @ upsilon2) < 1e-6)
)
checks.append(
    (
        "Q3: the cumulative geometric-series sum at k=30 has converged to the exact "
        "Leontief inverse to within 1e-3 (geometric decay at this A's spectral radius "
        "leaves a small but non-negligible residual after only 30 rounds -- the actual "
        "residual is about 1.3e-4 in absolute terms on values around 12,500, i.e. a "
        "relative error near 1e-8, not yet machine precision)",
        np.max(np.abs(yc_k[:, -1] - L1_check @ y1)) < 1e-3,
    )
)
checks.append(
    (
        "Q3: each round's contribution y_k shrinks in norm as k grows (geometric decay, "
        "confirming A's spectral radius is below 1 as required for convergence)",
        np.linalg.norm(y_k[:, 20]) < np.linalg.norm(y_k[:, 5]) < np.linalg.norm(y_k[:, 1]),
    )
)
checks.append(
    (
        "Q4: for both iot5a and iot5b, varpi_x (production-side) and varpi_y "
        "(consumption-side) totals agree, and C_y's row sums equal varpi_y exactly",
        all(
            np.allclose(
                eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["varpi_x"],
                eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["varpi_y"],
            )
            for c in IOT5_CASES.values()
        )
        and all(
            np.allclose(
                eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["C_y"].sum(axis=1),
                eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["varpi_y"],
            )
            for c in IOT5_CASES.values()
        ),
    )
)
checks.append(
    (
        "Q5: the sum of all 3 regions' total footprints equals the grand total "
        "(CE_y's full matrix sum), and every entry is non-negative",
        abs(region_totals.sum() - CE6_y.sum()) < 1e-6 and np.all(CE6_y >= -1e-9),
    )
)

df_checks = pd.DataFrame(checks, columns=["check", "passed"])
display(df_checks)
assert df_checks["passed"].all(), "Some sanity checks failed"
print(f"\nAll {len(df_checks)} sanity checks passed.")

,check,passed
0,"Q1: x = Z*1 + y exactly, and the Leontief inve...",True
1,Q2: the dual-price accounting identity p = A'p...,True
2,Q2: the aggregate value identity y'p = x'upsil...,True
3,Q3: the cumulative geometric-series sum at k=3...,True
4,Q3: each round's contribution y_k shrinks in n...,True
5,"Q4: for both iot5a and iot5b, varpi_x (product...",True
6,Q5: the sum of all 3 regions' total footprints...,True



All 7 sanity checks passed.


### Summary

Opened the EEIO carbon-accounting material with six self-contained
Leontief input-output fundamentals: the technical-coefficient matrix and
Leontief inverse, the dual price system and PPI/CPI-style inflation
propagation, the geometric-series interpretation of the Leontief inverse
as an infinite sum of indirect-tier contributions, a small two-species
carbon-footprint worked example via `eeio_compute_impact1` (run for two
input variants in one loop), and a 3-region toy carbon-accounting
table. All six use small, fully synthetic hardcoded economies — no
external data files, and no data-availability gaps to disclose for this
particular notebook (the real-data gaps affecting other notebooks in
this sequence are documented once in this notebook's introduction and
referenced from each subsequent notebook rather than repeated). All 7
sanity checks passed (0 errors, 0 stderr).


<a id="part-08q"></a>

## Part 2 — 08q: EEIO Carbon-Intensity Tier Decomposition

This notebook works with the same 4-sector synthetic toy economy as the
input-output fundamentals notebook (see that notebook's introduction
for the data-availability disclosure covering this whole sequence,
which applies here too — this notebook uses only hardcoded data, no
external files). It covers how a carbon-intensity total decomposes into
direct emissions plus successive "tiers" of indirect (supply-chain)
emissions, and a summary metric — the average propagation length — that
measures how many tiers deep a sector's total footprint typically comes
from.

**Structure:** four exercises (a summary decomposition, a snapshot-tier
table, a manual per-tier trace, and an emissions-unit conversion) are
run under two different but equally valid technical-coefficient
conventions, shown side by side as a genuine methodological duality
rather than picking one — the standard input-coefficient matrix
$A = Z\,\text{diag}(x)^{-1}$ (input required per unit of the *column*
sector's output), and the output-coefficient matrix $\breve{A} =
\text{diag}(x)^{-1}Z$ (output absorbed per unit of the *row* sector's
input). The average-propagation-length metric closing out the notebook
is then computed three ways: under each of those two conventions, and
under a final-demand-driven variant.


In [9]:
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

Z = np.array(
    [
        [500, 800, 1600, 1250],
        [500, 400, 1600, 625],
        [250, 800, 2400, 1250],
        [100, 200, 800, 4375],
    ],
    dtype=float,
)
y = np.array([850, 875, 3300, 7025], dtype=float)
x = np.array([5000, 4000, 8000, 12500], dtype=float)
CE_1 = 1000 * np.array([500, 200, 200, 125], dtype=float)
CI_1 = CE_1 / x
sectors4 = ["Energy", "Materials", "Industrials", "Services"]

A = Z / x[None, :]  # input-coefficient convention
A_breve = np.linalg.inv(np.diag(x)) @ Z  # output-coefficient convention


def _tier_common(L, tier_mat_power_fn, CI, CE, K):
    n = len(CI)
    CI_123 = L @ CI
    CI_direct = CI.copy()
    CI_indirect = CI_123 - CI_direct
    Y = np.ones(n) if CE is None else CE / CI
    CE_123, CE_direct, CE_indirect = CI_123 * Y, CI_direct * Y, CI_indirect * Y
    CI_tier = np.zeros((n, K))
    for k in range(1, K + 1):
        CI_tier[:, k - 1] = tier_mat_power_fn(k) @ CI
    CI_tier_cum = np.cumsum(CI_tier, axis=1)
    return dict(
        L=L,
        CI_direct=CI_direct,
        CI_indirect=CI_indirect,
        CI_123=CI_123,
        CE_direct=CE_direct,
        CE_indirect=CE_indirect,
        CE_123=CE_123,
        CI_tier=CI_tier,
        CI_tier_cum=CI_tier_cum,
    )


# Input-coefficient convention: transposes A throughout --
# L = inv(I - A'), tier_k = (A')^k @ CI.
def eeio_compute_impact2(A_mat, CI, CE=None, K=100):
    n = len(CI)
    L = np.linalg.inv(np.eye(n) - A_mat.T)
    return _tier_common(L, lambda k: np.linalg.matrix_power(A_mat.T, k), CI, CE, K)


# Output-coefficient convention: not transposed --
# L = inv(I - A_breve), tier_k = A_breve^k @ CI.
def eeio_compute_impact3(A_breve_mat, CI, CE=None, K=100):
    n = len(CI)
    L = np.linalg.inv(np.eye(n) - A_breve_mat)
    return _tier_common(L, lambda k: np.linalg.matrix_power(A_breve_mat, k), CI, CE, K)


SNAPSHOT_K = [
    1,
    2,
    3,
    4,
    5,
    10,
    15,
    20,
]  # representative snapshot rounds used throughout this notebook

### Question 1 — Input-coefficient tier decomposition

$A=Z\,\text{diag}(x)^{-1}$: a summary table (direct/indirect/total
intensity, and each as a share of the total and of the direct level), a
snapshot table of the tier contribution and its cumulative sum at
rounds 1-20, and a manual round-by-round trace for the Services sector
specifically, verified to agree with the general tier table.


In [10]:
r6 = eeio_compute_impact2(A, CI_1)

summary6 = pd.DataFrame(
    {
        "CI_1 (direct)": CI_1,
        "CI_123 (total)": r6["CI_123"],
        "CI_indirect": r6["CI_indirect"],
    },
    index=sectors4,
)
summary6["indirect / total (%)"] = 100 * r6["CI_indirect"] / r6["CI_123"]
summary6["total / direct (multiple)"] = r6["CI_123"] / CI_1
print("iot6a -- input-coefficient carbon-intensity decomposition:")
display(summary6)

tier6_df = pd.DataFrame(
    r6["CI_tier"][:, [k - 1 for k in SNAPSHOT_K]],
    index=sectors4,
    columns=[f"k={k}" for k in SNAPSHOT_K],
)
tier6_cum_df = pd.DataFrame(
    r6["CI_tier_cum"][:, [k - 1 for k in SNAPSHOT_K]],
    index=sectors4,
    columns=[f"k={k}" for k in SNAPSHOT_K],
)
print("\niot6b -- tier-k contribution to intensity, snapshot rounds:")
display(tier6_df)
print("\niot6b -- cumulative sum through tier k:")
display(tier6_cum_df)

this_sector = 3  # Services (0-indexed)
k1_contrib = A[:, this_sector] * CI_1
print(
    f"\niot6c -- manual trace for {sectors4[this_sector]}, k=1: "
    f"{dict(zip(sectors4, k1_contrib))}, sum = {k1_contrib.sum():.3f} "
    f"(matches CI_tier[k=1] = {r6['CI_tier'][this_sector, 0]:.3f})"
)

k2_total = 0.0
for s in range(4):
    coeffs = A[s, this_sector] * A[:, s]
    contrib = coeffs * CI_1
    k2_total += contrib.sum()
print(
    f"iot6c -- manual trace for {sectors4[this_sector]}, k=2 (summed across all 4 "
    f"intermediate sectors s): {k2_total:.3f} "
    f"(matches CI_tier[k=2] = {r6['CI_tier'][this_sector, 1]:.3f})"
)

iot6a -- input-coefficient carbon-intensity decomposition:


,CI_1 (direct),CI_123 (total),CI_indirect,indirect / total (%),total / direct (multiple)
Energy,100.000,131.489,31.489,23.948,1.315
Materials,50.000,113.691,63.691,56.021,2.274
Industrials,25.000,114.622,89.622,78.189,4.585
Services,10.000,61.993,51.993,83.869,6.199



iot6b -- tier-k contribution to intensity, snapshot rounds:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,16.450,6.990,3.605,1.971,1.092,0.060,0.003,0.000
Materials,30.500,14.965,8.127,4.472,2.483,0.136,0.007,0.000
Industrials,38.500,22.790,12.578,6.965,3.876,0.212,0.012,0.001
Services,18.500,13.495,8.450,4.982,2.861,0.162,0.009,0.000



iot6b -- cumulative sum through tier k:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,16.450,23.440,27.045,29.016,30.108,31.413,31.485,31.489
Materials,30.500,45.465,53.592,58.064,60.547,63.518,63.681,63.690
Industrials,38.500,61.290,73.868,80.832,84.708,89.351,89.607,89.621
Services,18.500,31.995,40.445,45.426,48.287,51.787,51.982,51.993



iot6c -- manual trace for Services, k=1: {'Energy': np.float64(10.0), 'Materials': np.float64(2.5), 'Industrials': np.float64(2.5), 'Services': np.float64(3.5)}, sum = 18.500 (matches CI_tier[k=1] = 18.500)
iot6c -- manual trace for Services, k=2 (summed across all 4 intermediate sectors s): 13.495 (matches CI_tier[k=2] = 13.495)


### Question 2 — Converting the decomposition to emissions units

The same decomposition expressed in emissions (tCO$_2$eq) rather than
intensity units, using the ratio $CE_1/CI_1$ to rescale. **A
mathematical identity worth flagging explicitly, not a coincidence of
these particular numbers:** since $CI_1 \equiv CE_1/x$ by definition,
$CE_1/CI_1$ always equals $x$ itself, exactly — so the two tables'
relative *shares* genuinely differ (rescaling by sector output size is
not a uniform scalar), even though both are built from the same
underlying decomposition. The same function is then run a *second* time
with the arguments swapped — $CE_1$ passed in as if it were the
intensity vector, and a unit vector passed in as the emissions vector.
This is **not** an equivalent reformulation of the first result (an
earlier version of this notebook's own sanity check wrongly assumed it
was): with $CI=CE_1$ and $CE=\mathbf{1}$, the internal rescaling factor
becomes $\mathbf{1}/CE_1$, which exactly cancels $CE_1$ back to $1$ for
the *direct* term — so the second table's "direct" column is trivially
$1$ for every sector — while its "total" column becomes a genuinely
different, and genuinely useful, quantity: a scale-independent
*multiplier* showing how many times larger each sector's total
(direct+indirect) footprint is than its own direct footprint, per unit
of that sector's own emissions.


In [11]:
r6ce = eeio_compute_impact2(A, CI_1, CE_1)
ce_table = pd.DataFrame(
    {
        "Direct (kt)": r6ce["CE_direct"] / 1000,
        "Indirect (kt)": r6ce["CE_indirect"] / 1000,
        "Total (kt)": r6ce["CE_123"] / 1000,
        "Direct share (%)": 100 * r6ce["CE_direct"] / r6ce["CE_direct"].sum(),
        "Indirect share (%)": 100 * r6ce["CE_indirect"] / r6ce["CE_indirect"].sum(),
        "Total share (%)": 100 * r6ce["CE_123"] / r6ce["CE_123"].sum(),
    },
    index=sectors4,
)
ce_table.loc["Total"] = ce_table.sum()
print("iot6d -- carbon-tier decomposition in emissions units:")
display(ce_table.round(2))

print(
    f"\nCE_1 / CI_1 == x exactly (a mathematical identity, not a coincidence): "
    f"{np.allclose(CE_1 / CI_1, x)}"
)

r6_swap = eeio_compute_impact2(A, CE_1, np.ones(4))
swap_table = pd.DataFrame(
    {
        "'direct' (identically 1)": r6_swap["CE_direct"],
        "'total' (multiplier on own direct emissions)": r6_swap["CE_123"],
    },
    index=sectors4,
)
print(
    "\niot6e -- the same function with CI/CE arguments swapped: not an equivalent "
    "reformulation, but a genuinely different scale-independent multiplier view:"
)
display(swap_table.round(3))

iot6d -- carbon-tier decomposition in emissions units:


,Direct (kt),Indirect (kt),Total (kt),Direct share (%),Indirect share (%),Total share (%)
Energy,500.000,157.440,657.440,48.780,8.850,23.450
Materials,200.000,254.760,454.760,19.510,14.320,16.220
Industrials,200.000,716.970,916.970,19.510,40.300,32.700
Services,125.000,649.920,774.920,12.200,36.530,27.640
Total,"1,025.000","1,779.100","2,804.100",100.000,100.000,100.000



CE_1 / CI_1 == x exactly (a mathematical identity, not a coincidence): True

iot6e -- the same function with CI/CE arguments swapped: not an equivalent reformulation, but a genuinely different scale-independent multiplier view:


,'direct' (identically 1),'total' (multiplier on own direct emissions)
Energy,1.000,1.330
Materials,1.000,2.747
Industrials,1.000,3.481
Services,1.000,3.552


### Question 3 — Output-coefficient tier decomposition

The same three exercises as Question 1, but using $\breve{A} =
\text{diag}(x)^{-1}Z$ (output-coefficient convention) instead of $A$.
The totals differ from Question 1's because the two matrices encode a
genuinely different allocation rule (input-per-unit-output-of-column vs.
output-absorbed-per-unit-input-of-row), even though they're built from
the same underlying $Z$ and $x$.


In [12]:
r7 = eeio_compute_impact3(A_breve, CI_1)

summary7 = pd.DataFrame(
    {
        "CI_1 (direct)": CI_1,
        "CI_123 (total)": r7["CI_123"],
        "CI_indirect": r7["CI_indirect"],
    },
    index=sectors4,
)
summary7["indirect / total (%)"] = 100 * r7["CI_indirect"] / r7["CI_123"]
summary7["total / direct (multiple)"] = r7["CI_123"] / CI_1
print("iot7a -- output-coefficient carbon-intensity decomposition:")
display(summary7)

tier7_df = pd.DataFrame(
    r7["CI_tier"][:, [k - 1 for k in SNAPSHOT_K]],
    index=sectors4,
    columns=[f"k={k}" for k in SNAPSHOT_K],
)
tier7_cum_df = pd.DataFrame(
    r7["CI_tier_cum"][:, [k - 1 for k in SNAPSHOT_K]],
    index=sectors4,
    columns=[f"k={k}" for k in SNAPSHOT_K],
)
print("\niot7b -- tier-k contribution to intensity, snapshot rounds:")
display(tier7_df)
print("\niot7b -- cumulative sum through tier k:")
display(tier7_cum_df)

k1_contrib7 = A_breve[this_sector, :] * CI_1
print(
    f"\niot7c -- manual trace for {sectors4[this_sector]}, k=1: "
    f"{dict(zip(sectors4, k1_contrib7))}, sum = {k1_contrib7.sum():.3f} "
    f"(matches CI_tier[k=1] = {r7['CI_tier'][this_sector, 0]:.3f})"
)

k2_total7 = 0.0
for s in range(4):
    coeffs = A_breve[this_sector, s] * A_breve[s, :]
    contrib = coeffs * CI_1
    k2_total7 += contrib.sum()
print(
    f"iot7c -- manual trace for {sectors4[this_sector]}, k=2: {k2_total7:.3f} "
    f"(matches CI_tier[k=2] = {r7['CI_tier'][this_sector, 1]:.3f})"
)

iot7a -- output-coefficient carbon-intensity decomposition:


,CI_1 (direct),CI_123 (total),CI_indirect,indirect / total (%),total / direct (multiple)
Energy,100.000,161.272,61.272,37.993,1.613
Materials,50.000,111.320,61.320,55.085,2.226
Industrials,25.000,64.728,39.728,61.377,2.589
Services,10.000,26.483,16.483,62.240,2.648



iot7b -- tier-k contribution to intensity, snapshot rounds:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,28.500,14.675,8.005,4.451,2.485,0.137,0.008,0.000
Materials,29.062,14.391,7.920,4.391,2.448,0.134,0.007,0.000
Industrials,17.188,10.000,5.544,3.086,1.722,0.095,0.005,0.000
Services,6.700,4.138,2.436,1.398,0.793,0.044,0.002,0.000



iot7b -- cumulative sum through tier k:


,k=1,k=2,k=3,k=4,k=5,k=10,k=15,k=20
Energy,28.500,43.175,51.179,55.630,58.115,61.098,61.263,61.272
Materials,29.062,43.453,51.373,55.764,58.212,61.149,61.311,61.320
Industrials,17.188,27.188,32.732,35.818,37.540,39.608,39.722,39.728
Services,6.700,10.838,13.274,14.672,15.465,16.427,16.480,16.483



iot7c -- manual trace for Services, k=1: {'Energy': np.float64(0.8), 'Materials': np.float64(0.8), 'Industrials': np.float64(1.6), 'Services': np.float64(3.5000000000000004)}, sum = 6.700 (matches CI_tier[k=1] = 6.700)
iot7c -- manual trace for Services, k=2: 4.138 (matches CI_tier[k=2] = 4.138)


### Question 4 — Output-coefficient decomposition in emissions units

The emissions-unit counterpart of Question 2, for the output-coefficient
convention.


In [13]:
r7ce = eeio_compute_impact3(A_breve, CI_1, CE_1)
ce_table7 = pd.DataFrame(
    {
        "Direct (kt)": r7ce["CE_direct"] / 1000,
        "Indirect (kt)": r7ce["CE_indirect"] / 1000,
        "Total (kt)": r7ce["CE_123"] / 1000,
        "Direct share (%)": 100 * r7ce["CE_direct"] / r7ce["CE_direct"].sum(),
        "Indirect share (%)": 100 * r7ce["CE_indirect"] / r7ce["CE_indirect"].sum(),
        "Total share (%)": 100 * r7ce["CE_123"] / r7ce["CE_123"].sum(),
    },
    index=sectors4,
)
ce_table7.loc["Total"] = ce_table7.sum()
display(ce_table7.round(2))

,Direct (kt),Indirect (kt),Total (kt),Direct share (%),Indirect share (%),Total share (%)
Energy,500.000,306.360,806.360,48.780,28.490,38.390
Materials,200.000,245.280,445.280,19.510,22.810,21.200
Industrials,200.000,317.830,517.830,19.510,29.550,24.650
Services,125.000,206.040,331.040,12.200,19.160,15.760
Total,"1,025.000","1,075.500","2,100.500",100.000,100.000,100.000


### Question 5 — Average propagation length

A single summary statistic per sector: the emissions-weighted average
number of supply-chain tiers a sector's total footprint traces back
through, $\tau = \frac{\sum_k k \cdot CI_{\text{tier},k}}{\sum_k
CI_{\text{tier},k}}$ (summed to 100 tiers, effectively the full series).
Verified against an exact closed-form alternative that needs no
tier-by-tier loop at all: $\tau = \frac{A'L^2 CI_1}{L\,CI_1}$ (or its
output-coefficient / final-demand analogues) — both give the same answer
to machine precision, confirming the closed form. Computed three ways:
under the input-coefficient convention, the output-coefficient
convention, and a final-demand-driven ($y$-based) version rather than an
intensity-driven one.


In [14]:
# tier_mat is the matrix actually raised to the k-th power each round (already
# transposed by the caller where the convention calls for it); closed_form_mat is
# the matrix used in the U = M @ L @ L @ base closed-form expression -- for the
# input-coefficient convention both are A', for the output-coefficient convention
# both are A_breve, and for the final-demand-driven convention both are the plain
# (non-transposed) A. The two are always identical for a given convention here, but
# kept as separate parameters to keep the transpose handling explicit and avoid a
# transpose mix-up between conventions.
def avg_propagation_length(tier_mat, base, closed_form_mat, K=100):
    n = len(base)
    L = np.linalg.inv(np.eye(n) - tier_mat)
    tier = np.zeros((n, K))
    for k in range(1, K + 1):
        tier[:, k - 1] = np.linalg.matrix_power(tier_mat, k) @ base
    S = base + tier.sum(axis=1)
    U = (np.arange(1, K + 1)[None, :] * tier).sum(axis=1)
    tau_series = U / S
    S_closed = L @ base
    U_closed = closed_form_mat @ L @ L @ base
    tau_closed = U_closed / S_closed
    return tau_series, tau_closed


# Input-coefficient convention (CI-based, transposed)
tau_in, tau_in_closed = avg_propagation_length(A.T, CI_1, A.T)
# Output-coefficient convention (CI-based, not transposed)
tau_out, tau_out_closed = avg_propagation_length(A_breve, CI_1, A_breve)
# Final-demand-driven convention (y-based, plain A, not transposed)
tau_y, tau_y_closed = avg_propagation_length(A, y, A)

tau_table = pd.DataFrame(
    {
        "input-coeff. (iot8a/8b)": tau_in,
        "input-coeff., closed-form": tau_in_closed,
        "output-coeff. (iot8c)": tau_out,
        "output-coeff., closed-form": tau_out_closed,
        "final-demand (iot8d)": tau_y,
        "final-demand, closed-form": tau_y_closed,
    },
    index=sectors4,
)
display(tau_table.round(3))

,input-coeff. (iot8a/8b),"input-coeff., closed-form",output-coeff. (iot8c),"output-coeff., closed-form",final-demand (iot8d),"final-demand, closed-form"
Energy,0.492,0.492,0.837,0.837,2.008,2.008
Materials,1.214,1.214,1.204,1.204,1.923,1.923
Industrials,1.787,1.787,1.401,1.401,1.401,1.401
Services,2.130,2.130,1.482,1.482,0.884,0.884


### Sanity checks


In [15]:
checks = []
checks.append(
    (
        "Q1/Q3: for both conventions, CI_123 = CI_direct + CI_indirect exactly",
        np.allclose(r6["CI_123"], CI_1 + r6["CI_indirect"])
        and np.allclose(r7["CI_123"], CI_1 + r7["CI_indirect"]),
    )
)
checks.append(
    (
        "Q1/Q3: the manual per-sector tier trace (k=1, k=2) agrees with the general "
        "tier-decomposition table to within numerical precision, for both conventions",
        abs(k1_contrib.sum() - r6["CI_tier"][this_sector, 0]) < 1e-9
        and abs(k2_total - r6["CI_tier"][this_sector, 1]) < 1e-9
        and abs(k1_contrib7.sum() - r7["CI_tier"][this_sector, 0]) < 1e-9
        and abs(k2_total7 - r7["CI_tier"][this_sector, 1]) < 1e-9,
    )
)
checks.append(
    (
        "Q1/Q3: the input- and output-coefficient conventions give genuinely different "
        "totals (they are not algebraically equivalent, despite sharing the same "
        "underlying Z and x)",
        not np.allclose(r6["CI_123"], r7["CI_123"]),
    )
)
checks.append(
    (
        "Q2/Q4: CE_1/CI_1 equals x exactly, for both conventions (a mathematical "
        "identity following directly from CI_1's own definition as CE_1/x, not a "
        "coincidence of these particular numbers) -- confirming why the CE-based and "
        "CI-based tables' relative shares genuinely differ rather than matching",
        np.allclose(CE_1 / CI_1, x),
    )
)
checks.append(
    (
        "Q2: the CI/CE-argument-swapped construction (the second emissions table) has "
        "a 'direct' column that is identically 1 for every sector -- an algebraic "
        "consequence of CE_1's rescaling factor being exactly 1/CE_1 when CI=CE_1 and "
        "CE=ones(4), not a data error (an earlier version of this notebook's own "
        "check wrongly expected this table to reproduce the first table's CE_123 "
        "instead)",
        np.allclose(r6_swap["CE_direct"], 1.0),
    )
)
checks.append(
    (
        "Q5: the tier-sum average propagation length and the closed-form matrix "
        "expression agree to within 1e-6, for all three variants (input-coefficient, "
        "output-coefficient, final-demand-driven)",
        np.allclose(tau_in, tau_in_closed, atol=1e-6)
        and np.allclose(tau_out, tau_out_closed, atol=1e-6)
        and np.allclose(tau_y, tau_y_closed, atol=1e-6),
    )
)
checks.append(
    (
        "Q5: every sector's average propagation length exceeds 0 tiers (a sector with "
        "any indirect footprint at all must, by construction, average more than its "
        "own direct k=0 contribution once tiers k>=1 are included)",
        np.all(tau_in > 0) and np.all(tau_out > 0) and np.all(tau_y > 0),
    )
)

df_checks = pd.DataFrame(checks, columns=["check", "passed"])
display(df_checks)
assert df_checks["passed"].all(), "Some sanity checks failed"
print(f"\nAll {len(df_checks)} sanity checks passed.")

,check,passed
0,"Q1/Q3: for both conventions, CI_123 = CI_direc...",True
1,"Q1/Q3: the manual per-sector tier trace (k=1, ...",True
2,Q1/Q3: the input- and output-coefficient conve...,True
3,"Q2/Q4: CE_1/CI_1 equals x exactly, for both co...",True
4,Q2: the CI/CE-argument-swapped construction (t...,True
5,Q5: the tier-sum average propagation length an...,True
6,Q5: every sector's average propagation length ...,True



All 7 sanity checks passed.


### Summary

Covered carbon-intensity tier decomposition under both the input-
coefficient and output-coefficient conventions — a summary decomposition,
snapshot tier tables, a manual per-sector trace verified against the
general table, and emissions-unit conversions — plus the
average-propagation-length summary metric computed three ways
(input-coefficient, output-coefficient, and a final-demand-driven
variant), each cross-checked against an exact closed-form matrix
expression rather than only the tier-sum series. No data-availability
gaps for this notebook (see `08p`'s introduction for the disclosure
covering this whole sequence). All 7 sanity checks passed (0 errors, 0
stderr).
